# 01 强化学习基本概念：从交互到 GridWorld

这一课先弄清楚强化学习在解决什么问题，再认识智能体、环境、状态、动作、策略、奖励和回合。最后才用 GridWorld 观察一条轨迹，并计算折扣回报。


## 本课的学习顺序

1. 先分清“预测一个答案”和“连续做决定”。
2. 看懂智能体与环境如何交互。
3. 区分策略、一步奖励和整段回报。
4. 用一个小网格把概念串起来。

读完后，应能用自己的话描述一次交互，并解释为什么两条都到达终点的路线仍可能有不同回报。先不讨论如何训练策略，也不实现算法。


## 1. 强化学习想解决什么问题？

想象一个机器人从房间入口出发，目标是找到出口。它每走一步都要决定方向；走了这一步，下一刻的位置就会改变，能选择的路线也跟着改变。

在你熟悉的监督学习中，通常先有一批固定样本及目标标签，模型学习输入与标签之间的关系。这里没有人逐格告诉机器人“正确动作”。机器人要在行动后观察结果，逐渐学会怎样做决定，才能让**整段过程**的结果更好。

所以，强化学习研究的核心是：面对一连串会相互影响的决策，怎样学习一个好的行动规则？


## 2. 一次交互是怎样发生的？

先认识两个角色：**智能体**（agent）做决定，**环境**（environment）接收动作并给出反馈。例如机器人是智能体，房间和移动规则属于环境。

在时刻 $t$，智能体看到当前状态 $s_t$，选动作 $a_t$；环境随后给出奖励 $r_{t+1}$ 和下一状态 $s_{t+1}$：

$$
(s_t,a_t)\xrightarrow{\text{环境}}(r_{t+1},s_{t+1})
$$

下一步，智能体在新状态中继续选择动作。注意分工：**智能体决定做什么，环境决定做完后发生什么**。这就是我们后面所有例子的基本循环。


## 3. 状态和动作分别是什么？

- **状态（state）** $s_t$：做决定时用来描述当前情况的信息。在下面的小网格里，先用位置“(行, 列)”表示。
- **动作（action）** $a_t$：智能体此刻能做的一种选择，例如“向右走”。
- **动作集合**：所有允许选择的动作。本例是上、下、左、右。

状态与动作不是同一种东西。“我在 (0, 1)”描述当前情况；“向右走”是准备采取的选择。采取动作后，环境才产生下一状态。现实任务中，动作也可能无法保证想要的结果，例如地面打滑；本课网格先采用确定的移动规则。


## 4. 什么是策略？

**策略（policy）**是智能体根据当前状态选择动作的规则。例如：“在上边一行就往右走；到最右列后往下走”。同一个策略要能在不同状态下给出选择。

它与“右、右、下、下”这一串已经发生的动作不同：前者是**决策规则**，后者是规则执行后产生的一条**轨迹**。一个策略也可以带有随机性，例如在某个格子以 70% 概率选“右”，以 30% 概率选“下”。

学习强化学习，最终希望找到能带来好结果的策略；这一课先学会辨认它，不急着训练它。


## 5. 一步奖励不等于最终目标

**奖励（reward）** $r_{t+1}$ 是采取一次动作后收到的反馈。它可以告诉智能体某一步有利还是不利，但一次决策的好坏还要看以后发生什么。

例如到达出口奖励 +10，每走一步付出 −1。绕远路也可能拿到 +10，却会多付几次 −1。因此，评价路线时不能只看最后一步；要看一整段奖励。

本例的目标是让智能体学到能获得更高**回报**的策略。回报是把后续多个奖励按规则合起来的量，具体算法稍后计算。


## 6. 回合与轨迹

从起点开始，智能体反复观察、行动、接收反馈，直到到达终点，这段过程称为一个**回合**（episode）。实际练习也常设置最大步数，避免一直走不出去。

回合中发生的一串“状态、动作、奖励、新状态”叫**轨迹**。一条轨迹告诉我们**这次实际走过了什么**；策略说明**一般情况下会怎样选择**。请先把这两个概念分开。

接下来把这些词放进同一个网格例子中。


## 7. GridWorld：把概念放进地图

| | 第 0 列 | 第 1 列 | 第 2 列 |
| --- | --- | --- | --- |
| 第 0 行 | 起点 S | · | · |
| 第 1 行 | · | 墙 # | · |
| 第 2 行 | · | · | 终点 G |

坐标使用“(行, 列)”。起点是 (0, 0)，终点是 (2, 2)，墙在 (1, 1)。机器人可选上、下、左、右；越界或撞墙时，位置不变，但仍消耗一步。普通动作的奖励是 −1，到达终点的那一步奖励为 +10。

在这个例子里，机器人是智能体；网格、墙、边界和奖励规则构成环境。位置是状态，方向是动作。“先沿顶行往右，再沿右列往下”可以作为一个策略。


## 8. 跟着一条轨迹逐步走

按“右、右、下、下”行动：

| 步数 | 原状态 | 动作 | 新状态 | 这一步的奖励 |
| --- | --- | --- | --- | ---: |
| 1 | (0, 0) | 右 | (0, 1) | −1 |
| 2 | (0, 1) | 右 | (0, 2) | −1 |
| 3 | (0, 2) | 下 | (1, 2) | −1 |
| 4 | (1, 2) | 下 | (2, 2) | +10 |

请逐行问自己：动作是谁选的？新状态和奖励是谁给的？到达 (2, 2) 后为什么停止？能回答这些问题，比背公式更重要。


## 9. 怎样从一步奖励得到回报？

这条轨迹的奖励序列是 $-1,-1,-1,10$。直接相加得到 7，但强化学习常用**折扣回报**，让较晚发生的奖励在当前时刻权重更小。折扣因子记为 $\gamma$，取值通常在 0 到 1 之间。

从起点看，四步的折扣回报是：

$$
G_0=r_1+\gamma r_2+\gamma^2r_3+\gamma^3r_4
$$

当 $\gamma=0.9$ 时：

$$
G_0=-1-0.9-0.81+7.29=4.58
$$

这里 $r_1$ 是**第一次动作之后**得到的奖励。$\gamma=1$ 时，对这条有限轨迹求普通总和，结果为 7。现阶段只需理解“每一步奖励”与“整段回报”的区别，以及晚到的奖励为什么乘上更高次幂。


In [ ]:
rewards = [-1, -1, -1, 10]  # 先手算轨迹，再改成练习路线的奖励序列
gamma = 0.9
sum(gamma**t * reward for t, reward in enumerate(rewards))


## 10. MDP 先认识名字即可

赵老师的第一课还会引出**马尔可夫决策过程**（MDP）。你现在可以把它看作描述“状态、动作、环境怎样转移、奖励怎样给出”的统一框架。

“马尔可夫”强调：如果当前状态包含了做下一步预测所需的信息，那么已知当前状态和动作时，就不必再依赖更早的整段历史。在本例中，位置足以确定下一位置和该步奖励；如果把“第 20 步必须结束”也当作环境规则，状态还需要记录已走步数。

这一课先建立直觉。状态转移概率、完整的 MDP 定义和贝尔曼方程留到后续课程。


## 11. 自己先算，再核对

从 (0, 0) 改走“右、下、右、下、下”。先不要运行代码：

1. 写出每一步的新位置。第二步撞墙后，状态是什么？
2. 写出五步的奖励序列。
3. 用 $\gamma=0.9$ 计算从起点看的折扣回报，再把奖励序列填入上面的代码单元核对。

这个小练习是在检查你能否区分**动作、状态转移、一步奖励与整段回报**。


## 12. 本课自检

- 智能体和环境各负责什么？
- 状态、动作、策略、轨迹四者有什么区别？
- 同样到达终点，绕远路为什么可能更差？
- 奖励和回报为什么不是同一个量？
- 当前状态要满足什么直觉条件，才能说“未来不需要依赖更早的历史”？

如果前四个问题能用自己的话解释清楚，就达到本课目标。下一课再继续研究策略如何评价和改进。


## 参考课程

本课的教学顺序参考了西湖大学赵世钰老师《强化学习的数学原理》的第一课：先讲 state、action、policy，再讲 reward、return、MDP。这里的文字、网格规则与手算练习是针对本学习工程重新编写的。

- [西湖大学赵世钰老师课程介绍](https://shiyuzhao.westlake.edu.cn/Teaching.htm)
- [B站课程合集：第1课基本概念 Part 1 与 Part 2](https://www.bilibili.com/video/BV1sd4y167NS/)
- [配套教材第一章：Basic Concepts](https://github.com/MathFoundationRL/Book-Mathematical-Foundation-of-Reinforcement-Learning/blob/main/3%20-%20Chapter%201%20Basic%20Concepts.pdf)
- [中国大学 MOOC 课程页](https://www.icourse163.org/course/XHUN-1470436188)
